In [ ]:
from glob import glob
import os

import numpy as np
import torch
from lightning import pytorch as L
from lightning.pytorch.utilities.model_summary import ModelSummary
from tifffile import imread, imwrite

from unet import LightningUNet
from visualization_utils import get_segmentation_visualization
from io_helpers import imsave_nowarnings

In [ ]:
# input paths: base in_path plus subdirectory for tiff files (with pattern)
in_path = "/Volumes/nn/Julia Vogtmann/Microscopy/25JV_006"
image_subdirectory = "tif"  # for resaved: "tif"
image_file_pattern = "*_ch1.tif"  # e.g. "*_ch0.tif" to only include images of one channel

# where to save results
out_subdirectory = "segmentation_nucleoli"
visualization_subdirectory = "vis"

# path to lightning model checkpoint
model_checkpoint = "/Users/david/Desktop/jurkat_nucleolin/unet_nucleolus_001/checkpoints/epoch=299-step=3300.ckpt"

# which class of the segmentation we are interested in
# None to save multiclass predictions as-is
# OR specific class index to save binary masks of just that class
# e.g. if model predicts: 0: background, 1: nucleus/cell, 2: nucleolus, use 2 to save segmentation of just nucleoli
class_to_save = 2

In [ ]:
# load model, show summary
model = LightningUNet.load_from_checkpoint(model_checkpoint).eval()

# trainer without checkpointing used for prediction
# NOTE: device can be set here, but should use GPU by default if available
trainer = L.Trainer(enable_checkpointing=False, logger=False)

ModelSummary(model, max_depth=3)

In [ ]:
# get input files
files = sorted(glob(os.path.join(in_path, image_subdirectory, image_file_pattern)))
files

In [ ]:
# create out directories
out_path = os.path.join(in_path, out_subdirectory)
vis_path = os.path.join(out_path, visualization_subdirectory)
os.makedirs(out_path, exist_ok=True)
os.makedirs(vis_path, exist_ok=True)

for in_file in files:

    # load data
    img = imread(in_file)
    # to torch, add two dummy dimensions (batch size, channels)
    img_t = torch.from_numpy(img).float()[:, torch.newaxis, torch.newaxis]

    print(f'predicting {in_file}...')
    
    # predict, concat batches
    with torch.no_grad():
        pred = trainer.predict(model, img_t)
        pred = torch.concat(pred)
        pred_labels = pred.argmax(1)

    # to numpy, pick smallest usable bit depth
    labels_np = pred_labels.numpy()
    out_dtype = np.uint16 if labels_np.max() > 255 else np.uint8
    labels_np.astype(out_dtype)

    # pick single class if necessary
    if class_to_save is not None:
        labels_np = (labels_np == class_to_save).astype(out_dtype)

    # simple visualization
    rgb_vis = get_segmentation_visualization(labels_np, img)
    
    # filenames for output
    out_file = os.path.join(out_path, os.path.basename(in_file).replace('.tif', '_segmented.tif'))
    vis_out_file = os.path.join(vis_path, os.path.basename(in_file).replace('.tif', '_segmented.png'))

    # save
    imwrite(out_file, labels_np, compression=5)
    imsave_nowarnings(vis_out_file, rgb_vis)